In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models  # Keras 레이어 / 모델 유틸

# Residual Block(ResNet의 기본 블록)을 만드는 함수
def residual_block(inputs, filters, downsample=False):
    x = inputs
    if downsample:
        strides = 2  # 공간 크기(H, W)를 절반으로 감소
    else:
        strides = 1  # 공간 크기 유지

    # 첫 번째 합성곱 층
    x = layers.Conv2D(filters, kernel_size=3, strides=strides, padding='same')(x)  # 필요시 다운샘플링
    x = layers.BatchNormalization()(x)  # 학습 안정화
    x = layers.ReLU()(x)

    # 두 번째 합성곱 층
    x = layers.Conv2D(filters, kernel_size=3, strides=1, padding='same')(x)  # 크기 유지
    x = layers.BatchNormalization()(x)

    # 스킵 연결
    if downsample:  # 입력과 출력의 shape을 맞춰줘야 하는 경우
        inputs = layers.Conv2D(filters, kernel_size=1, strides=2, padding='same')(inputs)  # 채널/크기 정렬
        inputs = layers.BatchNormalization()(inputs)

    x = layers.add([inputs, x])  # 스킵(입력) + 메인 경로 합산
    x = layers.ReLU()(x)
    return x

In [2]:
from tensorflow.keras.datasets import cifar10

# 데이터셋 로드 및 전처리
(train_images, train_labels), (test_images, test_labels) = cifar10.load_data()

# 픽셀 값 정규화
train_images, test_images = train_images / 255.0, test_images / 255.0  # 0~255 값을 0~1 범위로 정규화

c:\Users\Playdata\multimodal\multi_venv\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


In [3]:
# 입력층
inputs = layers.Input(shape=(32, 32, 3))

# 초기 합성곱 층
x = layers.Conv2D(64, kernel_size=7, strides=2, padding='same')(inputs)  # 7x7 Conv로 특징 추출 및 다운샘플링
x = layers.BatchNormalization()(x)
x = layers.ReLU()(x)
x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)  # 맥스풀링(공간 크기 감소)

# Residual Block 추가
x = residual_block(x, filters=64)  # 64채널
x = residual_block(x, filters=64)

x = residual_block(x, filters=128, downsample=True)  # 128채널 + 다운샘플링
x = residual_block(x, filters=128)

x = residual_block(x, filters=256, downsample=True)  # 256채널 + 다운샘플링
x = residual_block(x, filters=256)

x = layers.GlobalAveragePooling2D()(x)  # 각 채널별 평균값으로 1차원 배열로 변환
outputs = layers.Dense(10, activation='softmax')(x)  # 10개 클래스 확률 출력

# 모델 생성
model = models.Model(inputs=inputs, outputs=outputs)

# 모델 요약 출력
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 16, 16,    │      9,472 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 16, 16,    │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 16, 16,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 8, 8, 64)  │          0 │ re_lu[0][0]       │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 8, 8, 64)  │     36,928 │ max_pooling2d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 8, 64)  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 8, 8, 64)  │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 8, 8, 64)  │     36,928 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 8, 64)  │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 8, 8, 64)  │          0 │ max_pooling2d[0]… │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 8, 8, 64)  │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 8, 8, 64)  │     36,928 │ re_lu_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 8, 64)  │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 8, 8, 64)  │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 8, 8, 64)  │     36,928 │ re_lu_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 8, 64)  │        256 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 8, 8, 64)  │          0 │ re_lu_2[0][0],    │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_4 (ReLU)      │ (None, 8, 8, 64)  │          0 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 2,792,074 (10.65 MB)

 Trainable params: 2,787,594 (10.63 MB)

 Non-trainable params: 4,480 (17.50 KB)

In [4]:
# 모델 컴파일
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 모델 학습 (10% 검증데이터)
model.fit(train_images, train_labels, epochs=10, batch_size=64, validation_split=0.1)

Epoch 1/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 89s 119ms/step - accuracy: 0.5278 - loss: 1.3152 - val_accuracy: 0.4340 - val_loss: 1.7283
Epoch 2/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 134s 190ms/step - accuracy: 0.6671 - loss: 0.9395 - val_accuracy: 0.6540 - val_loss: 0.9985
Epoch 3/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 132s 175ms/step - accuracy: 0.7305 - loss: 0.7640 - val_accuracy: 0.6906 - val_loss: 0.8936
Epoch 4/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 121s 172ms/step - accuracy: 0.7752 - loss: 0.6391 - val_accuracy: 0.7054 - val_loss: 0.8452
Epoch 5/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 145s 176ms/step - accuracy: 0.8076 - loss: 0.5440 - val_accuracy: 0.6872 - val_loss: 0.9716
Epoch 6/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 123s 175ms/step - accuracy: 0.8429 - loss: 0.4482 - val_accuracy: 0.5144 - val_loss: 2.2195
Epoch 7/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 126s 152ms/step - accuracy: 0.8732 - loss: 0.3608 - val_accuracy: 0.6304 - val_loss: 1.3384
Epoch 8/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 64s 91ms/step - accuracy: 0.8991 - lo

In [5]:
# 모델 평가
test_loss, test_acc = model.evaluate(test_images, test_labels)
print('테스트 정확도:', test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.6937 - loss: 1.2090
테스트 정확도: 0.6937000155448914
